# 00. Environment and Data Audit

Runs dataset loader checks and reads paper hyperparameters from `hyper_parameter/*.json` so the manuscript setup tables and experiments share the same configuration source.

## Paper table targets

- Table 1: Benchmark datasets used in evaluation
- Tables 2-6: hyperparameter ranges, common settings, GA/PSO, H-ANFIS, and GRS-ANFIS hyperparameters
- Data dimension evidence for BCWD 80, Vowel 29, Spambase 57, Gisette 5000

This notebook does not import legacy result tables or emit hard-coded numeric rows. It runs the direct experiment backend and then displays CSV files generated in the same output tree.

In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "experiments" / "reproducible_paper_tables.py").exists():
            return path
    raise RuntimeError("Could not find project root")

PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_ROOT = PROJECT_ROOT / "output" / "reproducible_paper_tables"
PYTHON = PROJECT_ROOT / "scripts" / "python_with_local_deps.sh"
RUN_DIRECT_EXPERIMENTS = True
DATASETS = ["Breast_Cancer_Wisconsin_(Original)", "Vowel", "Spambase", "Gisette"]
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")

PROJECT_ROOT=/home/harp3133t/Research/03_Research/GRS-ANFIS_Github
OUTPUT_ROOT=/home/harp3133t/Research/03_Research/GRS-ANFIS_Github/output/reproducible_paper_tables


In [2]:
cmd = [str(PYTHON), "experiments/reproducible_paper_tables.py", "--task", "data-audit", "--output-root", str(OUTPUT_ROOT)]
cmd += ["--datasets", *DATASETS]
cmd += []
print(" ".join(cmd))
if RUN_DIRECT_EXPERIMENTS:
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    print("RUN_DIRECT_EXPERIMENTS=False: command not executed")

/home/harp3133t/Research/03_Research/GRS-ANFIS_Github/scripts/python_with_local_deps.sh experiments/reproducible_paper_tables.py --task data-audit --output-root /home/harp3133t/Research/03_Research/GRS-ANFIS_Github/output/reproducible_paper_tables --datasets Breast_Cancer_Wisconsin_(Original) Vowel Spambase Gisette
Loading /home/harp3133t/Research/03_Research/GRS-ANFIS_Github/data/gisette_train.data...
Loading /home/harp3133t/Research/03_Research/GRS-ANFIS_Github/data/gisette_train.labels...
Loading /home/harp3133t/Research/03_Research/GRS-ANFIS_Github/data/gisette_train.data...
Loading /home/harp3133t/Research/03_Research/GRS-ANFIS_Github/data/gisette_train.labels...
[data-audit] complete in 4.1s; output_root=/home/harp3133t/Research/03_Research/GRS-ANFIS_Github/output/reproducible_paper_tables


In [3]:
for rel in ['00_data_audit.csv',
    '00_hyperparameter_ranges.csv',
    '00_common_experiment_settings.csv',
    '00_ga_pso_hyperparameters.csv',
    '00_h_anfis_hyperparameters.csv',
    '00_grs_anfis_hyperparameters.csv',
    '00_ga_pso_selected_feature_resolution.csv',
    'paper_table_map.csv']:
    path = OUTPUT_ROOT / rel
    print(f"\n=== {rel} ===")
    if path.exists():
        display(pd.read_csv(path).head(20))
    else:
        print(f"missing: {path}")


=== 00_data_audit.csv ===


,dataset,display_dataset,n_samples,model_input_dim,n_classes,task_kind,source
0,Breast_Cancer_Wisconsin_(Original),Breast Cancer,699,80,2,binary,direct loader execution
1,Vowel,Vowel,990,29,11,multiclass,direct loader execution
2,Spambase,Spambase,4601,57,2,binary,direct loader execution
3,Gisette,Gisette,6000,5000,2,binary,direct loader execution



=== 00_hyperparameter_ranges.csv ===


,hyperparameter,role,typical_range
0,"lambda_primary, lambda_complementary",L1 sparsity penalty,1e-4 to 1e-1
1,"tau_primary, tau_complementary",Hard gate thresholds,"0.3 to 0.7, usually 0.5"
2,n,Membership functions per input,2 to 5
3,learning_rate_eta,Optimizer step size,1e-4 to 1e-2
4,max_epochs_s1_s2,Stage-wise maximum epochs,10 to 100



=== 00_common_experiment_settings.csv ===


...

,dataset,lr_primary,lr_complementary,primary_rules,complementary_rules,mf_per_feature,epochs_stage1,epochs_stage2,lambda_primary_s1,lambda_complementary_s2,weight_decay,primary_hard_epochs,complementary_hard_epochs,source_file
0,Breast Cancer,0.01,0.005,7,11,2,80,10,0.010,0.000,0.000010,5,5,hyper_parameter/best_GRS-ANFIS_HP.json
1,Vowel,0.10,0.005,7,11,2,80,10,0.010,0.000,0.000010,30,10,hyper_parameter/best_GRS-ANFIS_HP.json
2,Spambase,0.10,0.010,7,11,3,60,50,0.001,0.001,0.000001,50,50,hyper_parameter/best_GRS-ANFIS_HP.json
3,Gisette,0.01,0.010,7,11,2,30,30,0.010,0.001,0.000100,20,10,hyper_parameter/best_GRS-ANFIS_HP.json



=== 00_ga_pso_selected_feature_resolution.csv ===


,dataset,model,configured_selected_features,resolved_model_input_columns,unresolved_configured_features,unresolved_features_json,source_file
0,Breast Cancer,GA-ANFIS,5,44,0,[],hyper_parameter/best_GA-ANFIS_HP.json
1,Breast Cancer,PSO-ANFIS,7,63,0,[],hyper_parameter/best_PSO-ANFIS_HP.json
2,Vowel,GA-ANFIS,23,22,1,"[""Speaker_Number_nan""]",hyper_parameter/best_GA-ANFIS_HP.json
3,Vowel,PSO-ANFIS,19,16,3,"[""Train_or_Test_nan"", ""Speaker_Number_nan"", ""S...",hyper_parameter/best_PSO-ANFIS_HP.json
4,Spambase,GA-ANFIS,39,39,0,[],hyper_parameter/best_GA-ANFIS_HP.json
5,Spambase,PSO-ANFIS,31,31,0,[],hyper_parameter/best_PSO-ANFIS_HP.json
6,Gisette,GA-ANFIS,91,91,0,[],hyper_parameter/best_GA-ANFIS_HP.json
7,Gisette,PSO-ANFIS,87,87,0,[],hyper_parameter/best_PSO-ANFIS_HP.json



=== paper_table_map.csv ===


,notebook,paper_table_number_v6,paper_table_title_or_scope,evidence_block,generated_output
0,00_environment_and_data_audit.ipynb,Tables 1-6,Benchmark datasets and experiment hyperparamet...,dataset dimensions plus hyperparameters read f...,output/reproducible_paper_tables/00_*.csv
1,01_main_neuro_fuzzy_and_svm_experiments.ipynb,Tables 7-10,"Vowel, Spambase, Breast Cancer, and Gisette pe...","ANFIS-family, SVM, and GRS-ANFIS fold metrics",output/reproducible_paper_tables/01_main_neuro...
2,08_saved_model_interpretability_from_checkpoin...,Tables 7-10,Nauck/HFSi interpretability-index column for t...,loads saved fold checkpoints from Notebook 01 ...,output/reproducible_paper_tables/08_saved_mode...
3,02_tree_boosting_and_ebm_baselines.ipynb,Tables 7-10; Table A8,Modern tabular-baseline rows in the dataset pe...,RF/HGB/XGBoost/LightGBM/CatBoost/EBM fold metrics,output/reproducible_paper_tables/02_tabular_ba...
4,03_reviewer_table_package_rebuild.ipynb,"Tables 1, 7-17, A1-A9 package",Manuscript-ready CSV table package assembled f...,copies only tables generated by direct experim...,output/reproducible_paper_tables/manuscript_ta...
5,04_ablation_robustness_and_htsk_analysis.ipynb,Tables 13-14; Tables A1-A6,"Architecture, schedule, threshold, sparsity, H...",GRS threshold/lambda/architecture/schedule/fir...,output/reproducible_paper_tables/04_*.csv
6,05_complementary_boundary_and_error_recovery.i...,Table 12; Tables 15-17,Boundary-bin correction and representative Bre...,Q1 boundary-bin correction and BCWD case recov...,output/reproducible_paper_tables/05_boundary/*...
7,06_interpretability_shap_lime_grs_rule_path.ipynb,Table A7; supports Table 17,Explanation-form comparison and GRS rule-path ...,SHAP/LIME recomputation and GRS rule-path diag...,output/reproducible_paper_tables/06_interpreta...
8,07_statistical_tests_and_submission_tables.ipynb,Table A9; generated-table inventory,Paired statistics and final reproducibility in...,paired tests from generated fold-level results...,output/reproducible_paper_tables/07_*.csv
